# IITM Reinforcement Learning Course Project
## Industrial Inventory Control using Reinforcement Learning

This starter notebook demonstrates how to:

1. generate the assigned parameter variant from the official roll number;
2. create and inspect the Gymnasium environment;
3. convert between order quantities and the internal `MultiDiscrete` action;
4. interact with the environment for one episode;
5. organise experiments for five distinct RL techniques.

The notebook intentionally does **not** provide complete RL algorithms or a complete multi-seed evaluator. Those are part of the project work.

## 1. Setup

Keep the `industrial_inventory_env` folder in the same project directory as this notebook. Install the approved packages using:

```bash
pip install -r requirements.txt
```

In [ ]:
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from industrial_inventory_env import (
    IndustrialInventoryEnv,
    generate_student_config,
    public_config_summary,
)

np.set_printoptions(suppress=True)
print("Imports completed.")

## 2. Generate the assigned parameter variant

Enter the official roll number below. The same normalized roll number always generates the same variant for this project version. Do not use another student's roll number and do not modify the generation utility.

In [ ]:
ROLL_NUMBER = "ENTER_YOUR_ROLL_NUMBER"  # Replace this text once.

student_config = generate_student_config(ROLL_NUMBER)
config_summary = public_config_summary(student_config)

print("Assigned configuration generated successfully.")
for key, value in config_summary.items():
    print(f"{key}: {value}")

Record the generated `variant_id` and `config_fingerprint` in the notebook and brief report. The leaderboard evaluator will use separate common hidden configurations; it will not use student-supplied parameter values.

## 3. Create the environment

`scenario_mode="random"` permits stationary episodes and combinations of the declared seasonal, trend and temporary-shock characteristics. Episode parameters are generated deterministically from the reset seed.

In [ ]:
env = IndustrialInventoryEnv(
    student_config=student_config,
    scenario_mode="random",
    domain_randomization=True,
)

observation, info = env.reset(seed=2026)

print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
print("Variant:", info["variant_id"])
print("Scenario components:", info["episode_parameters"]["scenario_components"])
print("Episode demand multipliers:", info["episode_parameters"]["demand_multipliers"])
print("Episode initial inventory:", info["episode_parameters"]["initial_inventory"])
print("Episode delay probabilities:", info["episode_parameters"]["delay_probabilities"])

In [ ]:
for key, value in observation.items():
    print(f"{key:22s} shape={value.shape}, dtype={value.dtype}")
    print(value)
    print()

## 4. Action representation

The environment accepts internal indices `[0, ..., 10]` for each product. The leaderboard `run_policy(observation)` function must return actual quantities from `{0, 10, ..., 100}`. Use the supplied conversion utility during local interaction.

In [ ]:
order_quantities = [40, 20, 0]
action_indices = env.quantities_to_action_indices(order_quantities)

print("Actual order quantities:", order_quantities)
print("Internal action indices:", action_indices.tolist())
print("Converted back:", env.action_indices_to_quantities(action_indices).tolist())

## 5. One-step interaction example

At the beginning of the decision step, the policy receives the current observation and selects the order quantities. The environment then processes arrivals, capacity, the new order, demand and cost before returning the next observation.

In [ ]:
next_observation, reward, terminated, truncated, step_info = env.step(action_indices)

print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Demand:", step_info["demand"])
print("Daily costs:", step_info["costs"])
print("Next inventory:", next_observation["inventory"])

## 6. Demonstration policy

The following heuristic is included only to demonstrate the interaction loop. It is **not** one of the required RL techniques and should not be presented as an RL submission.

In [ ]:
def demonstration_policy(observation: dict) -> list[int]:
    """Simple deterministic inventory-position heuristic for demonstration."""
    inventory = np.asarray(observation["inventory"], dtype=float)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=float)
    inventory_position = inventory + pipeline.sum(axis=1)

    targets = np.asarray([110.0, 100.0, 120.0])
    required = np.maximum(targets - inventory_position, 0.0)
    quantities = np.clip(np.ceil(required / 10.0) * 10.0, 0.0, 100.0)
    return quantities.astype(int).tolist()

## 7. Run one deterministic episode

This example shows the Gymnasium loop and cost collection for a single seed. Students must build their own local evaluation process for comparing techniques across multiple seeds and scenario families.

In [ ]:
demo_env = IndustrialInventoryEnv(
    student_config,
    scenario_mode="random",
    domain_randomization=True,
)
observation, reset_info = demo_env.reset(seed=101)

records = []
while True:
    quantities = demonstration_policy(observation)
    action = demo_env.quantities_to_action_indices(quantities)
    observation, reward, terminated, truncated, info = demo_env.step(action)

    records.append(
        {
            "day": info["day"],
            "reward": reward,
            "daily_cost": info["costs"]["daily_total"],
            "episode_cost": info["costs"]["episode_total"],
            "inventory_p1": int(observation["inventory"][0]),
            "inventory_p2": int(observation["inventory"][1]),
            "inventory_p3": int(observation["inventory"][2]),
            "order_p1": int(quantities[0]),
            "order_p2": int(quantities[1]),
            "order_p3": int(quantities[2]),
        }
    )

    if terminated or truncated:
        break

one_episode_results = pd.DataFrame(records)
print("Episode days:", len(one_episode_results))
print("Total episode cost:", one_episode_results["daily_cost"].sum())
one_episode_results.head()

In [ ]:
one_episode_results.plot(
    x="day",
    y=["inventory_p1", "inventory_p2", "inventory_p3"],
    figsize=(10, 4),
    title="Inventory during the demonstration episode",
)
plt.ylabel("Units")
plt.show()

## 8. Build the local evaluation loop

Create a function that evaluates a supplied policy across multiple disclosed validation seeds and multiple declared scenario families. At minimum, record:

- average and standard deviation of total episode cost;
- holding, stockout, ordering and discarding costs;
- service level or unfulfilled demand;
- computational time;
- the seeds and scenario modes used.

Do not tune against one favourable seed. Keep the final public and private leaderboard episodes hidden and evaluator-controlled.

In [ ]:
# TODO: Implement your own evaluation function.
# Suggested signature:
#
# def evaluate_policy(policy, seeds, scenario_modes):
#     ...
#     return results_dataframe
#
# The function should call policy(observation), convert quantities to action indices,
# execute complete 50-day episodes and aggregate unscaled episode cost.


## 9. Track the five distinct techniques

Maintain a compact experiment table throughout the project. Use the exact technique categories announced by the instructors.

In [ ]:
technique_tracker = pd.DataFrame(
    {
        "technique": ["Technique 1", "Technique 2", "Technique 3", "Technique 4", "Technique 5"],
        "main_state_representation": ["", "", "", "", ""],
        "important_hyperparameters": ["", "", "", "", ""],
        "local_average_cost": [np.nan] * 5,
        "public_submission_id": ["", "", "", "", ""],
        "public_average_cost": [np.nan] * 5,
        "policy_file": ["", "", "", "", ""],
        "model_artifact": ["", "", "", "", ""],
    }
)
technique_tracker

## 10. Submission checks

For every policy file:

```bash
python policy_validation_tests.py path/to/policy_file.py
```

The final notebook must reproduce the training/export process and clearly map each frozen leaderboard submission to its technique, policy file and model artefact. The brief report should focus on what was implemented, what results were obtained and what was inferred.

11. Final results across all trained techniques

Seven techniques were implemented and evaluated on the local harness (evaluation.py / src/evaluation/) across three suites: Suite A (assigned configuration, 40 seeds x 5 scenario modes = 200 episodes), Suite B (7 legal configuration variants x 5 scenario modes x 6 seeds = 210 episodes, robustness check), and Suite C (a locked, held-out proxy of 125 episodes never used for hyperparameter tuning). The 5 scenario modes are the 4 declared demand families (stationary, seasonal, trend, shock) plus a local random/mixed testing mode that samples combinations of the declared components -- not five separately-declared families. Results below come from eval_results/phase2/phase2_suite_summary.csv, produced by run_phase2_evaluation.py.

phase2_summary = pd.read_csv("eval_results/phase2/phase2_suite_summary.csv")
suite_a_summary = (
    phase2_summary[phase2_summary["suite"] == "A"]
    .sort_values("mean_cost")
    .loc[:, ["policy", "mean_cost", "std_cost", "p95_cost", "mean_service_level", "minimum_scenario_service"]]
    .reset_index(drop=True)
)
suite_a_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ordered = suite_a_summary.sort_values("mean_cost", ascending=False)
axes[0].barh(ordered["policy"], ordered["mean_cost"], xerr=ordered["std_cost"], color="#4c72b0")
axes[0].set_xlabel("Suite A mean episode cost (lower is better)")
axes[0].set_title("Cost comparison (error bars = std across seeds)")

axes[1].barh(ordered["policy"], ordered["mean_service_level"], color="#55a868")
axes[1].axvline(0.95, color="red", linestyle="--", linewidth=1, label="95% service threshold")
axes[1].set_xlabel("Suite A mean service level")
axes[1].set_title("Service level comparison")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

12. Champion selection (final 5-technique portfolio)

Seven implemented techniques were narrowed to five using a lexicographic tournament, re-applied after a second improvement pass (see below):

Reject invalid or non-deterministic policies -- none rejected; all seven pass policy_validation_tests.py and the full audit in src/audit/checks.py.
Reject service level below 95% (an internal project selection rule adopted for this portfolio, not an official leaderboard qualification requirement) -- REINFORCE (90.9%) is rejected here.
Reject severe boundary failure -- none of the remaining six show one.
Minimize mean validation cost (Suite A) -- ranks: PPO (91,470) < DQN (96,871) < Neural SARSA (111,731) < A2C (114,669) < TD(lambda) (122,769) < Double DQN (126,999).
Top-5 cutoff: TD(lambda) (122,769) vs Double DQN (126,999) is a clear ~3.4% gap -- no tiebreak needed. TD(lambda) takes the 5th slot.
Final portfolio: PPO, DQN, Neural SARSA, A2C, TD(lambda). Double DQN remains fully implemented, tested and evaluated, but is not part of the final 5-technique submission this time.

Two feature pipelines are in use, side by side

35-dim legacy pipeline (training_utils/obs_wrapper.py, function flatten_observation): inventory, pipeline, 3/7-day demand means and std, day, capacity utilisation, inventory position, and a few derived ratios -- hand-engineered once during iteration 1 and never changed since. Used by PPO, DQN, A2C, and REINFORCE.
Representation B, 76-dim (src/features/{observation,engineered}.py): 38 raw normalized features (Representation A) + 38 further-engineered features (per-product inventory-position gap, lead-time-demand estimate, days-of-supply, demand trend; plus 8 global capacity/horizon features). Built later (iteration 2) specifically for the newer custom algorithms and fixed for a zero-padding bug in iteration 3. Used by Neural SARSA and Double DQN.
TD(lambda) uses neither -- it has its own 54-dim sparse, coarse-coded "band" discretization (src/algorithms/common/discretizer.py) purpose-built for its linear function approximator over a reduced 48-action catalogue.
These are three genuinely different, independently-verified pipelines (see the exported-vs-training-side parity check for every technique) -- not one canonical feature set reused everywhere.

Three notable engineering findings from the recovery/refinement phases:

PPO was the worst iteration-1 technique (391,365 cost) and became one of the best after a 6-configuration entropy-coefficient x learning-rate screen (which showed learning rate, not entropy, was the deciding factor) followed by promoting the winning configuration to 1.5M timesteps -- final cost 91,470, better than DQN.
Neural SARSA went through two rounds of work. Round 1 (single screening run) landed at 154,372 -- narrowly missing the original final 5. A follow-up audit found the engineered-feature pipeline had a real bug (early-episode demand means included zero-padded rows instead of masking them out, and "days of supply" used the wrong window). After fixing the features and running a proper 3-seed promotion of the winning screening config, Neural SARSA dropped to 111,731 cost / 99.3% service -- the 3rd-best technique overall, and a clear top-5 qualifier.
Double DQN was fixed twice. First, a real checkpoint-selection bug (EvalCallback's first evaluation firing while learning_starts was still active) was found and fixed, improving it from 173,958 to 153,822. Later, a cost-source diagnosis showed it was still buying service via brute-force over-ordering (54% capacity utilisation vs DQN's 26%). Retraining on the corrected engineered features with a longer replay warm-up cut it further to 126,999 cost / 98.8% service and eliminated discard cost entirely -- a big improvement, but not quite enough to beat TD(lambda) for the final 5th slot this time.
A2C refinement was attempted (12-configuration screen + 3-seed promotion of the top 2) but did not beat the original iteration-1 model, so the original was kept -- not every refinement attempt succeeds, and the honest outcome is reported rather than forced.

13. Final submission checks

Run policy_validation_tests.py against every policy file in the final portfolio before freezing the submission.

In [ ]:
import subprocess

final_portfolio = ["ppo", "dqn", "neural_sarsa", "a2c", "td_lambda"]

for technique in final_portfolio:
    policy_path = f"submissions/{technique}/policy.py"
    result = subprocess.run(
        [sys.executable, "policy_validation_tests.py", policy_path],
        capture_output=True,
        text=True,
    )
    status = "PASS" if result.returncode == 0 else "FAIL"
    print(f"[{status}] {technique:12s} -> {policy_path}")
    if result.returncode != 0:
        print(result.stdout[-1000:])
        print(result.stderr[-1000:])